In [ ]:
# this code generates a test set from the original dataset 
# with structures idfferents from the training set obtained with sampler.ipynb 

In [1]:
import os, glob, json
import numpy as np
import pandas as pd
from ase.io import read, write
from tqdm import tqdm

In [2]:
# --- 1. Locate provenance + filemap (universe of available structures)
desc_dir = "desc"
parquet = max(glob.glob(os.path.join(desc_dir, "*_provenance_*.parquet")), key=os.path.getmtime)
filemap = max(glob.glob(os.path.join(desc_dir, "*_filemap_*.json")), key=os.path.getmtime)
print("Using:", parquet, filemap)

meta = pd.read_parquet(parquet)
with open(filemap) as f:
    fmap = {int(k): v for k,v in json.load(f).items()}

universe = (
    meta.loc[:, ["file_id","struct_id"]]
        .drop_duplicates()
        .sort_values(["file_id","struct_id"])
)
universe["file_path"] = universe["file_id"].map(fmap)
print("Universe structures:", len(universe))

# --- 2. Collect all already used structures (from training manifests)
selected_dir = "selected/fullset/Random"
if not os.path.exists(selected_dir):
    os.makedirs(selected_dir)
manifests = glob.glob(os.path.join(selected_dir, "*_selected_manifest.csv"))
print("Found manifests:", manifests)

used = set()
for m in manifests:
    df = pd.read_csv(m)
    for fp, sid in zip(df["file_path"], df["struct_id"]):
        used.add((fp, int(sid)))
print("Already used structures:", len(used))

# --- 3. Exclude used
mask = ~universe.apply(lambda r: (r["file_path"], int(r["struct_id"])) in used, axis=1)
remaining = universe.loc[mask].reset_index(drop=True)
print("Remaining candidates:", len(remaining))

# --- 4. Choose validation set
#rng = np.random.default_rng(42)
rng = np.random.default_rng(12345)  # different seed from training set

# Option A: fixed number globally
n_val = 2000
sel_idx = rng.choice(remaining.index, size=min(n_val, len(remaining)), replace=False)
val_df = remaining.loc[sel_idx].sort_values(["file_path","struct_id"]).reset_index(drop=True)

# Option B: quota per file (e.g. 2 per file)
# val_df = remaining.groupby("file_path").apply(lambda g: g.sample(min(2, len(g)), random_state=42)).reset_index(drop=True)

val_df["n_atoms_hit"] = 0  # placeholder for schema compatibility

# --- 5. Save test manifest
test_manifest = os.path.join(selected_dir, "TEST_selected_manifest.csv")
val_df.to_csv(test_manifest, index=False)
print("Saved test manifest:", test_manifest, "with", len(val_df), "structures")

# load structures listed in TEST manifest
images = []
for fp, sid in tqdm(zip(val_df["file_path"], val_df["struct_id"]),
                    total=len(val_df), desc="Loading TEST structures"):
    images.append(read(fp, index=int(sid)))

# write merged trajectory and XYZ
traj_path = os.path.join(selected_dir, "TEST_selected.traj")
xyz_path  = os.path.join(selected_dir, "TEST_selected.xyz")
write(traj_path, images)
write(xyz_path, images)

print(f"Wrote {len(images)} frames:")
print("  .traj ->", traj_path)
print("  .xyz  ->", xyz_path)

Using: desc/SOAP_N-O-C_provenance_20250925-115459.parquet desc/SOAP_N-O-C_filemap_20250925-115459.json
Universe structures: 742046
Found manifests: ['selected/fullset/Random/Random_selected_manifest.csv']
Already used structures: 48409
Remaining candidates: 693637
Saved test manifest: selected/fullset/Random/TEST_selected_manifest.csv with 2000 structures


Loading TEST structures: 100%|██████████| 2000/2000 [24:03<00:00,  1.39it/s]


Wrote 2000 frames:
  .traj -> selected/fullset/Random/TEST_selected.traj
  .xyz  -> selected/fullset/Random/TEST_selected.xyz
